**Stephanie Nord**  
**DSC 630 Predictive Analytics**
**MovieLens Movie Recommendation System**
**Assignment 10.2**

## Project Overview

The objective of this project is to develop a movie recommendation system using the MovieLens small dataset. The system will allow a user to enter a movie they like and receive ten recommendations for other movies they may enjoy.

This project uses item-based collaborative filtering, which identifies relationships between items based on patterns in user ratings and generates recommendations from similarities among those rating patterns (Sarwar et al., 2001). The analysis will include data exploration and preparation, construction of a user-item ratings matrix, calculation of movie similarity, development of a recommendation function, and evaluation of the resulting recommendations.

## 1. Import Libraries

The analysis begins by importing the Python libraries required for data manipulation and numerical operations. Additional libraries will be introduced as they become necessary during development of the recommendation system.

In [1]:
# Import libraries for data manipulation and numerical operations
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

## 2. Load the Latest Small MovieLens Dataset

The MovieLens Latest Small dataset was obtained from GroupLens Research at the University of Minnesota. The dataset used in this analysis contains 100,836 ratings from 610 users across 9,742 movies (Harper & Konstan, 2015).

In [2]:
# Load the MovieLens datasets
movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")
tags = pd.read_csv("tags.csv")
links = pd.read_csv("links.csv")

## 3. Initial Data Inspection Summary

Before preparing the data for the recommendation system, each dataset was examined to understand its structure, variables, completeness, and relationship to the other MovieLens files.

### 3.1 Movies Dataset

In [3]:
# Display the first five rows of the movies dataset (preview)
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
# Inspect the dimensions and structure of the dataset (structure)
print("Movies dataset shape:", movies.shape)
movies.info()

Movies dataset shape: (9742, 3)
<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  9742 non-null   int64
 1   title    9742 non-null   str  
 2   genres   9742 non-null   str  
dtypes: int64(1), str(2)
memory usage: 626.1 KB


In [5]:
# Check the movies dataset for missing values and duplicate rows (data quality)
print("Missing values:")
print(movies.isna().sum())

print("\nDuplicate rows:", movies.duplicated().sum())

# Check whether movieId uniquely identifies each movie
print("\nUnique movie IDs:", movies["movieId"].nunique())
print("Duplicate movie IDs:", movies["movieId"].duplicated().sum())

Missing values:
movieId    0
title      0
genres     0
dtype: int64

Duplicate rows: 0

Unique movie IDs: 9742
Duplicate movie IDs: 0


### 3.2 Ratings Dataset

In [6]:
# Display the first five rows of the ratings dataset (preview)
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [7]:
# Inspect the dimensions and structure of the dataset (structure)
print("Ratings dataset shape:", ratings.shape)
ratings.info()

Ratings dataset shape: (100836, 4)
<class 'pandas.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


In [8]:
# Check the ratings dataset for missing values and duplicate rows (data quality)
print("Missing values:")
print(ratings.isna().sum())

print("\nDuplicate rows:", ratings.duplicated().sum())

# Check whether each user-movie combination uniquely identifies a rating
print(
    "\nDuplicate user-movie combinations:",
    ratings.duplicated(subset=["userId", "movieId"]).sum()
)

Missing values:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

Duplicate rows: 0

Duplicate user-movie combinations: 0


In [9]:
# Examine key characteristics of the ratings dataset (exploration)
print("Number of unique users:", ratings["userId"].nunique())
print("Number of unique movies rated:", ratings["movieId"].nunique())

print("\nRating values:")
print(ratings["rating"].value_counts().sort_index())

Number of unique users: 610
Number of unique movies rated: 9724

Rating values:
rating
0.5     1370
1.0     2811
1.5     1791
2.0     7551
2.5     5550
3.0    20047
3.5    13136
4.0    26818
4.5     8551
5.0    13211
Name: count, dtype: int64


### 3.3 Tags Dataset

In [10]:
# Display the first 5 rows of the tags dataset (preview)
tags.head()

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


In [11]:
# Inspect the dimensions and structure of the dataset (structure)
print("Tags dataset shape:", tags.shape)
tags.info()

Tags dataset shape: (3683, 4)
<class 'pandas.DataFrame'>
RangeIndex: 3683 entries, 0 to 3682
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   userId     3683 non-null   int64
 1   movieId    3683 non-null   int64
 2   tag        3683 non-null   str  
 3   timestamp  3683 non-null   int64
dtypes: int64(3), str(1)
memory usage: 151.7 KB


In [12]:
# Check the tags dataset for missing values and duplicate rows (data quality)
print("Missing values:")
print(tags.isna().sum())

print("\nDuplicate rows:", tags.duplicated().sum())

# Check whether each user movie tag combination is unique
print(
    "\nDuplicate user-movie-tag combinations:",
    tags.duplicated(subset=["userId", "movieId", "tag"]).sum()
)

Missing values:
userId       0
movieId      0
tag          0
timestamp    0
dtype: int64

Duplicate rows: 0

Duplicate user-movie-tag combinations: 0


### 3.4 Links Dataset

In [13]:
# Display the first five rows of the links dataset (preview)
links.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [14]:
# Inspect the dimensions and structure of the dataset (structure)
print("Links dataset shape:", links.shape)
links.info()

Links dataset shape: (9742, 3)
<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   movieId  9742 non-null   int64  
 1   imdbId   9742 non-null   int64  
 2   tmdbId   9734 non-null   float64
dtypes: float64(1), int64(2)
memory usage: 228.5 KB


In [15]:
# Check the links dataset for missing values and duplicate rows (data quality)
print("Missing values:")
print(links.isna().sum())

print("\nDuplicate rows:", links.duplicated().sum())

# Check whether movieId uniquely identifes each link record
print("\nUnique movie IDs:", links["movieId"].nunique())
print("Duplicate movie IDs:", links["movieId"].duplicated().sum())

# Check uniqueness of external database identifiers
print("\nDuplicate IMDB IDs:", links["imdbId"].duplicated().sum())
print(
    "Duplicate TMDB IDs:",
    links["tmdbId"].dropna().duplicated().sum())

Missing values:
movieId    0
imdbId     0
tmdbId     8
dtype: int64

Duplicate rows: 0

Unique movie IDs: 9742
Duplicate movie IDs: 0

Duplicate IMDB IDs: 0
Duplicate TMDB IDs: 1


In [16]:
# Identify records with duplicated TMDb identifiers
duplicate_tmdb = links[
    links["tmdbId"].notna()
    & links["tmdbId"].duplicated(keep=False)
    ]
duplicate_tmdb

,movieId,imdbId,tmdbId
4169,6003,290538,4912.0
9106,144606,270288,4912.0


#### Initial Data Inspection Summary

The four MovieLens datasets were inspected to understand their structure, completeness, and relationships. The `movies` dataset contains 9,742 uniquely identified movies with no missing values or duplicate rows. The `ratings` dataset contains 100,836 ratings from 610 users and includes 9,724 uniquely rated movies. Each rating is uniquely identified by the combination of `userId` and `movieId`.

The `tags` dataset contains user-generated descriptive tags associated with movies, while the `links` dataset maps MovieLens movie identifiers to IMDb and TMDb identifiers. The core `movies` and `ratings` datasets were complete and free of duplicate records. Eight TMDb identifiers were missing, and one TMDb identifier was associated with two MovieLens records; however, these external identifiers are not required for the collaborative filtering model.

## 4. Exploratory Analysis of User Ratings

### 4.1 Rating Distribution

In [17]:
# Summarize the distribution of movie ratings
ratings["rating"].describe()

count    100836.000000
mean          3.501557
std           1.042529
min           0.500000
25%           3.000000
50%           3.500000
75%           4.000000
max           5.000000
Name: rating, dtype: float64

### 4.2 Movie Rating Frequency

In [18]:
# Count the number of ratings per movie
ratings_per_movie = ratings.groupby("movieId")["rating"].count()

ratings_per_movie.head()

movieId
1    215
2    110
3     52
4      7
5     49
Name: rating, dtype: int64

In [19]:
# Idenitfy the movies with the most ratings
ratings_per_movie.sort_values(ascending=False).head(10)

movieId
356     329
318     317
296     307
593     279
2571    278
260     251
480     238
110     237
589     224
527     220
Name: rating, dtype: int64

In [20]:
# Add movie titles to the rating counts
movie_rating_counts = movies.merge(
    ratings_per_movie.rename("rating_count"),
    on="movieId",
    how="left",
)

movie_rating_counts.head()   

,movieId,title,genres,rating_count
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,215.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,110.0
2,3,Grumpier Old Men (1995),Comedy|Romance,52.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,7.0
4,5,Father of the Bride Part II (1995),Comedy,49.0


In [21]:
# Identify movies with no ratings (Boolean Indexing)
unrated_movies = movie_rating_counts[
    movie_rating_counts["rating_count"].isna()
]
print("Number of unrated movies:", len(unrated_movies))
unrated_movies

Number of unrated movies: 18


,movieId,title,genres,rating_count
816,1076,"Innocents, The (1961)",Drama|Horror|Thriller,NaN
2211,2939,Niagara (1953),Drama|Thriller,NaN
2499,3338,For All Mankind (1989),Documentary,NaN
2587,3456,"Color of Paradise, The (Rang-e khoda) (1999)",Drama,NaN
3118,4194,I Know Where I'm Going! (1945),Drama|Romance|War,NaN
4037,5721,"Chosen, The (1981)",Drama,NaN
4506,6668,"Road Home, The (Wo de fu qin mu qin) (1999)",Drama|Romance,NaN
4598,6849,Scrooge (1970),Drama|Fantasy|Musical,NaN
4704,7020,Proof (1991),Comedy|Drama|Romance,NaN
5020,7792,"Parallax View, The (1974)",Thriller,NaN


In [22]:
# Replace missing rating counts with zero
movie_rating_counts["rating_count"] = (
    movie_rating_counts["rating_count"]
    .fillna(0)
    .astype(int)
)

movie_rating_counts.head()

,movieId,title,genres,rating_count
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,215
1,2,Jumanji (1995),Adventure|Children|Fantasy,110
2,3,Grumpier Old Men (1995),Comedy|Romance,52
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,7
4,5,Father of the Bride Part II (1995),Comedy,49


In [23]:
 # Display the top 10 movies with the most ratings
movie_rating_counts.sort_values(
    by="rating_count",
    ascending=False
).head(10)

,movieId,title,genres,rating_count
314,356,Forrest Gump (1994),Comedy|Drama|Romance|War,329
277,318,"Shawshank Redemption, The (1994)",Crime|Drama,317
257,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,307
510,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,279
1939,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,278
224,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,251
418,480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,238
97,110,Braveheart (1995),Action|Drama|War,237
507,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi,224
461,527,Schindler's List (1993),Drama|War,220


### 4.3 Rating Count vs. Average Rating

In [24]:
# Calculate rating count and average rating for each movie
movie_rating_stats = (
    ratings.groupby("movieId")["rating"]
    .agg(["count", "mean"])
    .reset_index()
)

movie_rating_stats.head()

,movieId,count,mean
0,1,215,3.920930
1,2,110,3.431818
2,3,52,3.259615
3,4,7,2.357143
4,5,49,3.071429


In [25]:
# Add movie titles to the rating statistics
movie_rating_stats = movie_rating_stats.merge(
    movies[["movieId", "title"]],
    on="movieId",
    how="left",
)

# Round average ratings to two decimal points
movie_rating_stats["mean"] = movie_rating_stats["mean"].round(2)

movie_rating_stats.head()

,movieId,count,mean,title
0,1,215,3.92,Toy Story (1995)
1,2,110,3.43,Jumanji (1995)
2,3,52,3.26,Grumpier Old Men (1995)
3,4,7,2.36,Waiting to Exhale (1995)
4,5,49,3.07,Father of the Bride Part II (1995)


In [26]:
# Display 10 movies with the highest number of ratings (which movies have most rating evidence)
top_by_count = movie_rating_stats.sort_values(
    by="count",
    ascending=False,
).head(10)

top_by_count

,movieId,count,mean,title
314,356,329,4.16,Forrest Gump (1994)
277,318,317,4.43,"Shawshank Redemption, The (1994)"
257,296,307,4.20,Pulp Fiction (1994)
510,593,279,4.16,"Silence of the Lambs, The (1991)"
1938,2571,278,4.19,"Matrix, The (1999)"
224,260,251,4.23,Star Wars: Episode IV - A New Hope (1977)
418,480,238,3.75,Jurassic Park (1993)
97,110,237,4.03,Braveheart (1995)
507,589,224,3.97,Terminator 2: Judgment Day (1991)
461,527,220,4.22,Schindler's List (1993)


In [27]:
# Display 10 movies with the highest average rating (which movies have highest average score)
top_by_mean = movie_rating_stats.sort_values(
    by = "mean",
    ascending=False,
).head(10)

top_by_mean

,movieId,count,mean,title
48,53,2,5.0,Lamerica (1994)
7232,74226,1,5.0,"Dream of Light (a.k.a. Quince Tree Sun, The) (..."
8855,134004,1,5.0,What Love Is (2007)
3931,5537,1,5.0,Satin Rouge (2002)
3918,5513,1,5.0,Martin Lawrence Live: Runteldat (2002)
8807,131610,1,5.0,Willy/Milly (1986)
8810,131724,1,5.0,The Jinx: The Life and Deaths of Robert Durst ...
8819,132153,1,5.0,Buzzard (2015)
5414,25887,1,5.0,Tales of Manhattan (1942)
8859,134095,1,5.0,My Love (2006)


In [28]:
# Filter for movies with at least 50 ratings (boolean indexing)
popular_movies = movie_rating_stats[
    movie_rating_stats["count"] >= 50
]

# Display the 10 highest rated movies among sufficiently rated movies
top_rated_popular = popular_movies.sort_values(
    by="mean",
    ascending=False,
).head(10)

top_rated_popular

,movieId,count,mean,title
277,318,317,4.43,"Shawshank Redemption, The (1994)"
659,858,192,4.29,"Godfather, The (1972)"
974,1276,57,4.27,Cool Hand Luke (1967)
2224,2959,218,4.27,Fight Club (1999)
602,750,97,4.27,Dr. Strangelove or: How I Learned to Stop Worr...
686,904,84,4.26,Rear Window (1954)
921,1221,129,4.26,"Godfather: Part II, The (1974)"
913,1213,126,4.25,Goodfellas (1990)
6298,48516,107,4.25,"Departed, The (2006)"
6693,58559,149,4.24,"Dark Knight, The (2008)"


#### Exploratory Data Analysis Summary

The comparison between rating count and average rating demonstrates the importance of considering the number of observations supporting an average score. Several movies with a perfect 5.0 average had only one or two ratings, making their scores less representative of broader user preferences. Applying an exploratory minimum threshold of 50 ratings produced a ranking supported by substantially more user feedback. This threshold is used here for exploratory comparison rather than as a universal statistical cutoff. The result illustrates why recommendation and ranking systems should consider both preference scores and the amount of evidence supporting those scores.

# 5. Data Preparation for the Recommendation System

### 5.1 Create the Movie-User Rating Matrix

In [29]:
# Create a movie-user rating matrix from the ratings data
movie_user_matrix =  ratings.pivot(
    index = "movieId",  # Each row should represent a movie
    columns = "userId",  # Each column should represent a user
    values = "rating"  # The ratings go inside each cell
)

movie_user_matrix.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
movieId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,NaN,NaN,4.0,NaN,4.5,NaN,NaN,NaN,...,4.0,NaN,4.0,3.0,4.0,2.5,4.0,2.5,3.0,5.0
2,NaN,NaN,NaN,NaN,NaN,4.0,NaN,4.0,NaN,NaN,...,NaN,4.0,NaN,5.0,3.5,NaN,NaN,2.0,NaN,NaN
3,4.0,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN


### 5.2 Evaluate Matrix Sparsity

In [30]:
# Calculate the total number of possible ratings in the matrix
total_possible_ratings = (
    movie_user_matrix.shape[0] * movie_user_matrix.shape[1]
)

# Count the number of ratings actually observed
observed_ratings = movie_user_matrix.notna().sum().sum() # Gives total of non-missing cells in matrix

# Calculate the percentage of the matrix that contains ratings
density = observed_ratings / total_possible_ratings * 100
sparsity = 100 - density

print("Matrix shape:", movie_user_matrix.shape)
print("Total possible ratings:", total_possible_ratings)
print("Observed_ratings:", observed_ratings)
print(f"Matrix density: {density:.2f}%")
print(f"Matrix sparsity: {sparsity:.2f}%")

Matrix shape: (9724, 610)
Total possible ratings: 5931640
Observed_ratings: 100836
Matrix density: 1.70%
Matrix sparsity: 98.30%


The movie-user matrix is highly sparse, with only 1.70% of all possible user-movie combinations containing observed ratings and 98.30% containing no rating. This level of sparsity is expected in recommendation data because individual users typically rate only a small portion of the available movies. Therefore, a sparse matrix representation is appropriate for storing and processing the rating data efficiently.

### 5.3 Convert to a Sparse Matrix Representation

In [31]:
# Replace missing ratings with zero for sparse matrix conversion
movie_user_filled = movie_user_matrix.fillna(0)

movie_user_filled.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
movieId,,,,,,,,,,,,,,,,,,,,,
1,4.0,0.0,0.0,0.0,4.0,0.0,4.5,0.0,0.0,0.0,...,4.0,0.0,4.0,3.0,4.0,2.5,4.0,2.5,3.0,5.0
2,0.0,0.0,0.0,0.0,0.0,4.0,0.0,4.0,0.0,0.0,...,0.0,4.0,0.0,5.0,3.5,0.0,0.0,2.0,0.0,0.0
3,4.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0


In [32]:
# Convert the filled movie user matrix to Compressed Sparse Row (CSR) format
movie_user_sparse = csr_matrix(movie_user_filled.values)

print("Sparse matrix shape:", movie_user_sparse.shape)
print("Stored nonzero values:", movie_user_sparse.nnz)

Sparse matrix shape: (9724, 610)
Stored nonzero values: 100836


# 6. Build the Item-Based Collaborative Filtering Model

### 6.1 Measure Movie Similarity

Cosine similarity was used to measure similarity between movie-rating vectors. 
This metric compares the orientation of two vectors rather than their absolute 
magnitude, making it useful for identifying movies with similar patterns of 
user ratings. Item-based collaborative filtering uses these similarities 
between items to identify potential recommendations (Sarwar et al., 2001).

In [33]:
# Calculate movie-to-movie cosine similarity
movie_similarity = cosine_similarity(movie_user_sparse)

print("Similarity matrix shape:", movie_similarity.shape)

Similarity matrix shape: (9724, 9724)


### 6.2 Inspect Similar Movies

In [34]:
# Create a mapping between movie IDs and matrix row positions
movie_id_to_index = {
    movie_id: index
    for index, movie_id in enumerate(movie_user_matrix.index)
}

toy_story_index = movie_id_to_index[1]

print("Toy story matrix index:", toy_story_index)

Toy story matrix index: 0


In [35]:
# Retrieve Toy Story's similarity scores with all movies
toy_story_similarities = movie_similarity[toy_story_index]

toy_story_similarities[:10].round(3)

array([1.   , 0.411, 0.297, 0.036, 0.309, 0.376, 0.277, 0.132, 0.233,
       0.396])

In [36]:
# Get movie positions sorted by similarity from highest to lowest
sorted_indices = np.argsort(toy_story_similarities)[::-1]

sorted_indices[:10]

array([   0, 2353,  418,  615,  224,  314,  322,  910,  546,  963])

In [37]:
# Display the highest similarity scores
toy_story_similarities[sorted_indices[:10]].round(3)

array([1.   , 0.573, 0.566, 0.564, 0.557, 0.547, 0.541, 0.541, 0.539,
       0.534])

In [38]:
# Remove Toy Story and keep the 10 most similar movie positions (boolean filtering)
recommended_indices = sorted_indices[
    sorted_indices != toy_story_index
][:10]

recommended_indices

array([2353,  418,  615,  224,  314,  322,  910,  546,  963,  968])

In [39]:
# Convert the recommended matrix positions back to MovieLens movie IDs
recommended_movie_ids = movie_user_matrix.index[recommended_indices]

recommended_movie_ids

Index([3114, 480, 780, 260, 356, 364, 1210, 648, 1265, 1270], dtype='int64', name='movieId')

In [40]:
# Retrieve movie titles for the 10 most similar movies
toy_story_recommendations = movies[
    movies["movieId"].isin(recommended_movie_ids)
][["movieId", "title", "genres"]]

toy_story_recommendations

,movieId,title,genres
224,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi
314,356,Forrest Gump (1994),Comedy|Drama|Romance|War
322,364,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX
418,480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller
546,648,Mission: Impossible (1996),Action|Adventure|Mystery|Thriller
615,780,Independence Day (a.k.a. ID4) (1996),Action|Adventure|Sci-Fi|Thriller
911,1210,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi
964,1265,Groundhog Day (1993),Comedy|Fantasy|Romance
969,1270,Back to the Future (1985),Adventure|Comedy|Sci-Fi
2355,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy


In [41]:
# Create a ranked table of Toy Story recommendations
toy_story_results = pd.DataFrame({
    "movieId": recommended_movie_ids,
    "similarity": toy_story_similarities[recommended_indices]
})

toy_story_results

,movieId,similarity
0,3114,0.572601
1,480,0.565637
2,780,0.564262
3,260,0.557388
4,356,0.547096
5,364,0.541145
6,1210,0.541089
7,648,0.538913
8,1265,0.534169
9,1270,0.530381


In [42]:
# Add movie titles and genres to ranked recommendations
toy_story_results = toy_story_results.merge(
    movies[["movieId", "title", "genres"]],
    on="movieId",
    how="left"
)

toy_story_results["similarity"] = (
    toy_story_results["similarity"].round(3)
)

toy_story_results

,movieId,similarity,title,genres
0,3114,0.573,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
1,480,0.566,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller
2,780,0.564,Independence Day (a.k.a. ID4) (1996),Action|Adventure|Sci-Fi|Thriller
3,260,0.557,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi
4,356,0.547,Forrest Gump (1994),Comedy|Drama|Romance|War
5,364,0.541,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX
6,1210,0.541,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi
7,648,0.539,Mission: Impossible (1996),Action|Adventure|Mystery|Thriller
8,1265,0.534,Groundhog Day (1993),Comedy|Fantasy|Romance
9,1270,0.530,Back to the Future (1985),Adventure|Comedy|Sci-Fi


#### Collaborative Filtering Model Interpretation

The item-based collaborative filtering model produced intuitively reasonable recommendations for Toy Story (1995). Toy Story 2 (1999) received the highest cosine similarity score, while several other highly rated mainstream films also appeared among the nearest items. Because the model is based on user-rating patterns rather than genre metadata, recommended movies do not necessarily share the same genres. Instead, they reflect similarity in how users interacted with and rated the movies.

# 7. Build the Recommendation Function

In [43]:
# Create a lookup table from movie title to Movielens movie ID
title_to_movie_id = dict(
    zip(movies["title"], movies["movieId"])
)

In [44]:
def recommend_movies(movie_title, n=10):
    """
    Recommend movies using item-based collaborative filtering.

    Parameters
    ----------
    movie_title : str
        Title of a movie contained in the MovieLens dataset.
    n : int, default=10
        Number of recommendations to return.

    Returns
    -------
    pandas.DataFrame
        Ranked movie recommendations with cosine similarity scores.
    """

    # Check whether the entered title exists in the dataset
    if movie_title not in title_to_movie_id:
        return f"Movie '{movie_title}' was not found in the dataset."

    # Retrieve the MovieLens movie ID
    movie_id = title_to_movie_id[movie_title]

    # Check whether the movie has rating data
    if movie_id not in movie_id_to_index:
        return f"Movie '{movie_title}' has no user ratings and cannot be used for collaborative filtering."

    # Find the movie's row position in the similarity matrix
    movie_index = movie_id_to_index[movie_id]

    # Retrieve similarity scores with all other movies
    similarity_scores = movie_similarity[movie_index]

    # Sort movie positions from highest to lowest similarity
    sorted_indices = np.argsort(similarity_scores)[::-1]

    # Remove the input movie itself and keep the top n matches
    recommended_indices = sorted_indices[
        sorted_indices != movie_index
    ][:n]

    # Convert matrix positions back to MovieLens movie IDs
    recommended_movie_ids = movie_user_matrix.index[
        recommended_indices
    ]

    # Create ranked recommendation results
    recommendations = pd.DataFrame({
        "movieId": recommended_movie_ids,
        "similarity": similarity_scores[recommended_indices]
    })

    # Add movie titles and genres
    recommendations = recommendations.merge(
        movies[["movieId", "title", "genres"]],
        on="movieId",
        how="left"
    )

    # Round similarity values for readability
    recommendations["similarity"] = (
        recommendations["similarity"].round(3)
    )

    return recommendations

## 7.1 Test the Recommendation Function

In [45]:
recommend_movies("Grumpier Old Men (1995)")

,movieId,similarity,title,genres
0,3450,0.448,Grumpy Old Men (1993),Comedy
1,762,0.447,Striptease (1996),Comedy|Crime
2,788,0.445,"Nutty Professor, The (1996)",Comedy|Fantasy|Romance|Sci-Fi
3,736,0.436,Twister (1996),Action|Adventure|Romance|Thriller
4,5,0.418,Father of the Bride Part II (1995),Comedy
5,95,0.410,Broken Arrow (1996),Action|Adventure|Thriller
6,65,0.407,Bio-Dome (1996),Comedy
7,708,0.403,"Truth About Cats & Dogs, The (1996)",Comedy|Romance
8,7,0.403,Sabrina (1995),Comedy|Romance
9,141,0.400,"Birdcage, The (1996)",Comedy


In [46]:
recommend_movies("Chalet Girl (2011)")

"Movie 'Chalet Girl (2011)' has no user ratings and cannot be used for collaborative filtering."

In [47]:
recommend_movies("Forrest Gump (1994)")

,movieId,similarity,title,genres
0,318,0.713,"Shawshank Redemption, The (1994)",Crime|Drama
1,480,0.688,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller
2,296,0.686,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
3,110,0.643,Braveheart (1995),Action|Drama|War
4,593,0.639,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller
5,150,0.628,Apollo 13 (1995),Adventure|Drama|IMAX
6,2571,0.610,"Matrix, The (1999)",Action|Sci-Fi|Thriller
7,500,0.608,Mrs. Doubtfire (1993),Comedy|Drama
8,527,0.602,Schindler's List (1993),Drama|War
9,589,0.601,Terminator 2: Judgment Day (1991),Action|Sci-Fi


## 8. Model Evaluation and Limitations

The recommendation function produced different similarity patterns depending on the input movie. Forrest Gump (1994) generated relatively strong cosine similarities, with several recommendations above 0.60, while Grumpier Old Men (1995) produced weaker similarities around 0.40–0.45. This suggests that some movies have more strongly aligned user-rating patterns than others. The recommendations are based on collaborative behavior rather than movie metadata, so genre matches are not guaranteed.

### 8.1 Cold-Start Limitation

Chalet Girl (2011) exists in the MovieLens catalog but has no user ratings. Because item-based collaborative filtering depends on user-rating vectors, the system cannot calculate meaningful similarity for this movie. This is an example of the item cold-start problem. A content-based or hybrid recommender using genres, tags, or other movie attributes could help address this limitation.

### 8.2 Sparsity

The movie-user matrix is 98.30% sparse, meaning most users have rated only a small fraction of the available movies. High sparsity can weaken similarity estimates because two movies may have limited overlap in the users who rated them. Although sparse matrix representations make this type of data computationally efficient to store and process, sparsity remains an important limitation when estimating relationships among movies.

### 8.3 Popularity Bias

Popular movies receive more ratings and therefore tend to have richer interaction histories. This can make their similarity estimates more stable and may cause them to appear more frequently in recommendations than less frequently rated movies. Earlier exploratory analysis also showed that average rating alone can be misleading when based on only one or two observations.



### 8.4 Input Validation

In [48]:
# Test the function with a title that is not in the dataset
recommend_movies("Definitely Not a Real Movie (2026)")

"Movie 'Definitely Not a Real Movie (2026)' was not found in the dataset."

# 9. Conclusion

This project developed an item-based collaborative filtering recommender system using the MovieLens small dataset. User ratings were transformed into a movie-user matrix in which each movie was represented by its pattern of ratings across users. Because the resulting matrix was 98.30% sparse, it was converted to a compressed sparse row (CSR) representation before cosine similarity was used to compare movie-rating vectors. The final recommendation function allows a user to enter a movie contained in the dataset and returns the ten movies with the most similar user-rating patterns.

The results demonstrate both the usefulness and limitations of collaborative filtering. Recommendations for movies such as Forrest Gump (1994) showed relatively strong similarity among user-rating patterns, while other movies produced weaker relationships. The analysis also demonstrated the item cold-start problem when Chalet Girl (2011) could not be used to generate recommendations because it had no user ratings. High matrix sparsity and popularity bias represent additional limitations because similarity estimates are more reliable for movies with richer interaction histories.

Overall, this project demonstrates how user behavior can be transformed into numerical vectors and compared mathematically to generate recommendations without relying directly on movie content. Future improvements could incorporate genres, tags, or other metadata into a hybrid recommender system, which could provide recommendations for movies with limited or nonexistent rating histories.

# 10. References

GroupLens Research. (2018). MovieLens latest datasets. University of Minnesota.
MovieLens datasets

Harper, F. M., & Konstan, J. A. (2015). The MovieLens datasets: History and context. ACM Transactions on Interactive Intelligent Systems, 5(4), Article 19.
https://doi.org/10.1145/2827872

Sarwar, B., Karypis, G., Konstan, J., & Riedl, J. (2001). Item-based collaborative filtering recommendation algorithms. In Proceedings of the 10th International Conference on World Wide Web (pp. 285–295). Association for Computing Machinery.
https://doi.org/10.1145/371920.372071